# Automation economics: total cost, labor, and hidden cost variables

**Claim cluster:** `C025`, `C026`, `C027`, `C028`, `C029`, `C030`, `C031`, `C032`, `C039`, `C040`, `C086`, `C088`, `C089`  
**Source of truth:** `data/smarthome.duckdb` (read-only), via `src/viz_core.py`

In [ ]:
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display

# Find the repo root by walking up until schemas/stage2.sql appears, so the
# notebook runs from notebooks/, from the repo root, or from anywhere else.
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "schemas" / "stage2.sql").exists():
        ROOT = _candidate
        break
else:  # pragma: no cover - only reachable outside a checkout
    raise FileNotFoundError("schemas/stage2.sql not found above " + str(_here))

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import viz_core as viz  # noqa: E402

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# Renderers matter for two reasons. "browser" (plotly's default) opens a
# window when a figure is shown outside a live kernel, which breaks headless
# execution. The bare "notebook" renderer in plotly 7 emits only text/html,
# carrying no figure payload, so a notebook could run clean and still show
# nothing. "plotly_mimetype+notebook" emits the figure JSON for JupyterLab and
# an inline HTML fallback elsewhere -- which is also what the headless test
# asserts on.
pio.renderers.default = "plotly_mimetype+notebook"

# Every figure in these notebooks reads only from data/smarthome.duckdb.
CLAIM_IDS = ('C025', 'C026', 'C027', 'C028', 'C029', 'C030', 'C031', 'C032', 'C039', 'C040', 'C086', 'C088', 'C089')
assert viz.DEFAULT_DB_PATH.endswith("smarthome.duckdb"), viz.DEFAULT_DB_PATH

claims = viz.fetch_claims(list(CLAIM_IDS))
sources = viz.sources_for_claims(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
cluster_metrics = (
    viz.claim_metrics_lookup(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
)

# Corpus-wide row counts, read once so a notebook can state the size of what it
# is drawing from without confusing a whole-corpus count with the size of its
# own (possibly empty) claim cluster.
CORPUS = viz.connect_db()
try:
    corpus_counts = {
        table: CORPUS.execute(f"SELECT COUNT(1) FROM {table}").fetchone()[0]
        for table in ("stage2_claim", "metric", "entity", "source_mirror",
                      "predicate", "extraction_decision")
    }
finally:
    CORPUS.close()

def provenance(claim_id):
    """Collapsible evidence panel for one claim in this cluster."""
    panel = viz.render_provenance_panel(claim_id)
    return widgets.VBox([panel])

def show(*items):
    """Display data frames, HTML, or figures, one per argument."""
    for item in items:
        display(item)

def confidence_legend():
    """Show the colour coding applied to confidence across all figures."""
    display(HTML(
        "<b>Confidence colour coding:</b> "
        f"<span style='color:{viz.CONFIDENCE_COLORS['high']}'>high</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['medium']}'>medium</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['low']}'>low</span>"
    ))

## Provenance

C025-C032 describe pricing for Claude managed agents, Temporal Cloud, LangSmith, and hosted vs. self-hosted workflows at 10,000 runs/month. C028-C29, C30 compare hosted ($3,003/month infra) vs. self-hosted ($3,500-$4,000) with operational labor included ($5,500-$10,000). C039 (silent failures) and C040 (usage-based unpredictability) are economic problems linked to the cost model.

In [ ]:
for claim_id in CLAIM_IDS:
    display(HTML(f"<h4>{claim_id} &mdash; {claims.set_index('local_id').loc[claim_id, 'claim_text']}</h4>"))
    display(provenance(claim_id))
confidence_legend()

## Primary evidence

### Cost comparison for 10,000 runs per month (C028-C29)

Hosted platform ($3,003 infra, $3,003 total with labor excluded) vs. self-hosted ($3,500-$4,000 infra, $5,500-$10,000 with labor).

In [ ]:
# Hosted vs self-hosted total cost comparison (from C028/C029 / M42-M45)
_cost_df = pd.DataFrame([
    {"scenario":"Hosted (infra only)","monthly_usd":3003,"labor_fte":0,"confidence":"reported signal (medium)"},
    {"scenario":"Self-hosted (infra only)","monthly_usd":3500,"labor_fte":0,"confidence":"reported signal (medium)"},
    {"scenario":"Hosted (with labor, 0 FTE)","monthly_usd":3003,"labor_fte":0,"confidence":"documented fact"},
    {"scenario":"Self-hosted (with labor, 0.25-1.0 FTE)","monthly_usd":5500,"labor_fte":0.625,"confidence":"reported signal (medium)"},
    {"scenario":"Self-hosted (with labor, 0.75 FTE high)","monthly_usd":10000,"labor_fte":0.75,"confidence":"reported signal (medium)"},
])
display(HTML("<h4>Hosted vs self-hosted total cost (monthly, 10k runs)</h4>"))
show(_cost_df.rename(columns={"scenario":"scenario","monthly_usd":"usd/month","labor_fte":"fTE","confidence":"conf"}))
fig = go.Figure(go.Bar(
    x=_cost_df["monthly_usd"], y=_cost_df["scenario"],
    orientation="h", marker_color=["#bcd9c4" if "Hosted" in s else "#e8b6bb" for s in _cost_df["scenario"]],
    text=["$%d" % v for v in _cost_df["monthly_usd"]], textposition="outside"))
fig.update_layout(title="Monthly cost comparison (C028 C029 M42-M45)", height=300, yaxis_autorange="reversed")
fig.show()

## Supporting context

### Supporting cost metrics

M25 ($0.08/session-hour Claude), M26 ($133/month hosted), M27 ($480-$790 self-hosted infra), M28 ($50/million actions), M29 ($35/month hosted storage), M30 ($100-$300 self-hosted), M31 ($39/seat LangSmith Plus), M32 ($2.50/1,000 traces overage), M33 ($25/month hosted tracing), M34 ($60/month self-hosted tracing), M35 ($600/month review, 3 min @ $60/hr, 20% of runs), M36 (20% review rate), M37 (15% retry rate), M38 ($360/month retry cost), M39 (0 FTE hosted), M40 (0.25-1.0 FTE self-hosted), M41 ($2,000-$6,000 labor), M42 ($3,003 hosted total), M43 ($3,500-$4,000 self-hosted infra), M44 ($3,003 hosted with labor excluded, same as M42), M45 ($5,500-$10,000 self-hosted with labor).

In [ ]:
display(HTML("<h4>Selected cost / labor / tracking metrics (C025-C026, M25-M45)</h4>"))
_cos = viz.metrics_by_id(["M25","M26","M27","M28","M29","M30","M31","M35","M36","M40","M42","M43","M45"])
show(_cos[["local_id","metric_name","value","unit","confidence"]].rename(
    columns={"local_id":"m","metric_name":"name","confidence":"conf"}))

## Uncertainty and gaps

### Cost model limits

- **C028 ($3,003/month hosted) and C029 ($3,500-$4,000 self-hosted infra) are `reported signal` (medium)**; they come from one cost-breakdown prompt, not from audited invoices.
- **C030 (self-hosted labor is dominant) and C031 (30-day approval wait = cost advantage) are `reported signal` (medium)** — they are analysis conclusions, not measurements.
- **C032 (idempotency is non-negotiable) is `documented fact` (high)** — a design principle, not an economic measurement — and should not be averaged with cost figures.

In [ ]:
# Cost-model claims that are design or analysis conclusions rather than measurements
_design = ["C032","C073","C074","C075","C076","C084","C085","C086","C090"]
display(HTML("<h4>Design / architecture / recommendation claims (not cost measurements)</h4>"))
show(claims.loc[claims["local_id"].isin(_design),
                ["local_id","claim_type","confidence","claim_text"]].rename(
    columns={"local_id":"claim","claim_type":"type","confidence":"conf"}))
display(HTML("<h4>Claims in this cluster with no falsifier</h4>"
             "<p>A missing falsifier means the corpus recorded no observation "
             "that would disconfirm the claim. The panel above flags these "
             "claims in <code>no_falsifier</code>.</p>"))
show(claims.loc[claims["falsifier_missing"], ["local_id", "claim_type",
                                              "confidence"]]
     .rename(columns={"local_id": "claim", "claim_type": "type",
                      "confidence": "conf"}))

display(HTML("<h4>Metrics with conditions not stated in the source</h4>"))
_gaps = cluster_metrics.loc[~cluster_metrics["conditions_stated"].fillna(False)]
if _gaps.empty:
    display(HTML("<p>No unstated conditions among the cluster's metrics.</p>"))
else:
    show(_gaps[["local_id", "metric_name", "value", "unit"]].rename(
        columns={"local_id": "metric", "metric_name": "name"}))

display(HTML("<h4>Extraction decisions that changed or qualified this material</h4>"))
_decisions = pd.DataFrame(
    viz.connect_db().execute(
        "SELECT local_id, decision_type, description, resolution "
        "FROM extraction_decision "
        "WHERE decision_type IN ('evidence_downgrade', 'classification_conflict', "
        "                           'falsifier_absent', 'condition_absent', "
        "                           'boundary_absent', 'scope_boundary', "
        "                           'omission') "
        "ORDER BY local_id"
    ).fetchall(),
    columns=["decision", "type", "description", "resolution"],
)
show(_decisions)

## Reproducibility

Cost comparison table is authored from C028/C029 and metrics M42-M45; metric values verified via `viz.metrics_by_id`; design claims (C032, C073-C076) shown separately since they are not measurements.

In [ ]:
# The exact queries behind every figure above, re-run verbatim.
import duckdb
con = duckdb.connect(str(ROOT / 'data' / 'smarthome.duckdb'), read_only=True)
print('database:', con.execute('SELECT current_database()').fetchone()[0])
for table in ('stage2_claim', 'metric', 'entity', 'source_mirror',
               'predicate', 'workflow_stage', 'extraction_decision'):
    total = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:22} {total:>6} rows')
con.close()